# Agentic AI - Self-Correcting Data Extraction with Reflection & Pydantic
## بهبود و اعتبارسنجی خودکار داده‌ها با الگوی بازاندیشی (Reflection)

###  هدف آموزشی (Learning Outcome)
در این نوت‌بوک یاد می‌گیرید چگونه با استفاده از **الگوی بازاندیشی (Reflection Pattern)** و دریافت **بازخورد خارجی (External Feedback)** از کتابخانه `Pydantic`:
1. داده‌های نامنظم متنی را به فرمت ساختاریافته استخراج کنید (نسخه V1).
2. خطاهای اعتبارسنجی را توسط محیط واقعی پایتون کشف کنید.
3. خطاهای کشف‌شده را به مدل ارسال کنید تا خروجی خود را نقد و اصلاح کند (نسخه V2).
4. یک پایپ‌لاین خودکار و مطمئن برای تضمین ۱۰۰٪ سلامت داده‌ها بسازید.

## ۱. آماده‌سازی و اتصال به مدل

In [9]:
import os
import json
import re
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator, ValidationError
from openai import OpenAI

#===========================================
# بارگذاری متغیرهای محیطی از فایل .env
#===========================================
load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY", "")
BASE_URL = os.getenv("OPENAI_BASE_URL")
MODEL_NAME = os.getenv("MODEL_NAME", "gpt-4o-mini")
#===========================================
# راه‌اندازی کلاینت
#===========================================
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print("✅ کلاینت با موفقیت راه‌اندازی شد. مدل فعال:", MODEL_NAME)

✅ کلاینت با موفقیت راه‌اندازی شد. مدل فعال: gpt-4o-mini


## ۲. تعریف ساختار استاندارد داده (Pydantic Schema)
در این بخش، کلاسی تعریف می‌کنیم که قوانین داده معتبر را مشخص می‌کند:
* نام مشتری نباید خالی باشد.
* ایمیل باید دارای علامت `@` و فرمت معتبر باشد.
* مبلغ فاکتور باید **حتماً مثبت** باشد.
* تعداد اقلام باید **حداقل ۱** باشد.

In [2]:
class UserInvoice(BaseModel):
    customer_name: str = Field(..., description="نام مشتری")
    email: str = Field(..., description="ایمیل معتبر")
    total_amount: float = Field(..., description="مبلغ کل فاکتور (باید مثبت باشد)")
    items_count: int = Field(..., ge=1, description="تعداد اقلام (حداقل ۱)")

    @field_validator("email")
    def check_email(cls, v):
        if "@" not in v or "." not in v:
            raise ValueError(f"ایمیل '{v}' نامعتبر است و باید شامل @ و دامنه باشد.")
        return v

    @field_validator("total_amount")
    def check_positive_price(cls, v):
        if v <= 0:
            raise ValueError(f"مبلغ فاکتور ({v}) نمی‌تواند صفر یا منفی باشد.")
        return v

print("اسکیما و شروط اعتبارسنجی با موفقیت تعریف شدند.")

✅ اسکیما و شروط اعتبارسنجی با موفقیت تعریف شدند.


## ۳. تعریف متن ورودی و استخراج اولیه (V1)
یک متن نامنظم داریم که دارای دو خطای ظریف است:
1. ایمیل به جای `@` با کلمه `[at]` نوشته شده است.
2. مبلغ به دلیل بستانکاری قبلی با عدد منفی `-250` بیان شده است.

In [10]:
sample_text = """
سلام، سفارش آقای آرش کیانی ثبت شد.
ایمیل ارتباطی ایشان arash[at]gmail_com است.
ایشان ۲ عدد کتاب خریداری کردند به مبلغ کل منفی ۲۵۰ هزار تومان (-250.0)
به خاطر طلب قبلی‌شان، اما ارزش کل خرید همان ۲۵۰ مثبت است.
"""

print("متن ورودی:\n", sample_text.strip())

متن ورودی:
 سلام، سفارش آقای سهراب سپهری ثبت شد.
ایمیل ارتباطی ایشان sohrab[at]gmail_com است.
ایشان ۲ عدد کتاب خریداری کردند به مبلغ کل منفی ۲۵۰ هزار تومان (-250.0)
به خاطر طلب قبلی‌شان، اما ارزش کل خرید همان ۲۵۰ مثبت است.


### ۳.۱. تابع تولید نسخه اول (V1 Generator)
مدل تلاش اولیه خود را برای استخراج JSON انجام می‌دهد.

In [11]:
def generate_v1(text: str) -> str:
    prompt = f"""
    Extract the following JSON fields from the text:
    - customer_name (string)
    - email (string)
    - total_amount (float)
    - items_count (int)

    Text: {text}

    Return ONLY strict JSON.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    # حذف تگ‌های مارک‌داون در صورت وجود
    if raw.startswith("```json"):
        raw = raw[7:-3]
    elif raw.startswith("```"):
        raw = raw[3:-3]
    return raw.strip()

v1_output = generate_v1(sample_text)
print("خروجی اولیه (V1):\n", v1_output)

خروجی اولیه (V1):
 {
    "customer_name": "سهراب سپهری",
    "email": "sohrab@gmail.com",
    "total_amount": 250.0,
    "items_count": 2
}


## ۴. اعتبارسنجی در محیط واقعی (Execution & External Feedback)
اکنون خروجی `v1_output` را به کلاس `UserInvoice` در پایتون می‌دهیم تا ببینیم آیا قوانین را پاس می‌کند یا خیر.

In [5]:
def validate_json(json_str: str):
    try:
        data = json.loads(json_str)
        valid_obj = UserInvoice(**data)
        return True, valid_obj.model_dump_json(indent=2)
    except (ValidationError, json.JSONDecodeError) as e:
        return False, str(e)

is_valid, feedback = validate_json(v1_output)

if not is_valid:
    print("خروجی V1 خطای اعتبارسنجی دارد (External Feedback):\n")
    print(feedback)
else:
    print("خروجی بدون خطا تایید شد:", feedback)

✅ خروجی بدون خطا تایید شد: {
  "customer_name": "سهراب سپهری",
  "email": "sohrab@gmail.com",
  "total_amount": 250.0,
  "items_count": 2
}


## ۵. بازاندیشی و اصلاح خروجی (Reflection & Refinement to V2)
در این گام، بازخورد خطای دقیق Pydantic را به مدل می‌دهیم تا خودش را اصلاح کند.

In [6]:
def refine_with_reflection(text: str, bad_json: str, error_feedback: str) -> str:
    prompt = f"""
    You are a data validation refiner (Reflection Pattern).

    The previous JSON failed validation.

    Original Text: {text}
    Failed JSON (V1): {bad_json}
    Validation Errors: {error_feedback}

    Fix the errors (e.g. fix email format to standard @ and make amount positive) and return ONLY valid JSON.
    """
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = response.choices[0].message.content.strip()
    if raw.startswith("```json"):
        raw = raw[7:-3]
    elif raw.startswith("```"):
        raw = raw[3:-3]
    return raw.strip()

v2_output = refine_with_reflection(sample_text, v1_output, feedback)
print("خروجی اصلاح‌شده پس از بازاندیشی (V2):\n", v2_output)

🔁 خروجی اصلاح‌شده پس از بازاندیشی (V2):
 {
    "customer_name": "سهراب سپهری",
    "email": "sohrab@gmail.com",
    "total_amount": 250.0,
    "items_count": 2
}


### ۵.۱. اعتبارسنجی نهایی خروجی V2
حال خروجی اصلاح‌شده را مجدداً به Pydantic می‌دهیم تا از صحت قطعی آن مطمئن شویم.

In [7]:
is_valid_v2, final_result = validate_json(v2_output)

if is_valid_v2:
    print(" تایید نهایی: داده‌ها ۱۰۰٪ معتبر و منطبق با اسکیما شدند!\n")
    print(final_result)
else:
    print("هنوز خطا وجود دارد:", final_result)

🎉 تایید نهایی: داده‌ها ۱۰۰٪ معتبر و منطبق با اسکیما شدند!

{
  "customer_name": "سهراب سپهری",
  "email": "sohrab@gmail.com",
  "total_amount": 250.0,
  "items_count": 2
}


## ۶. تجمیع کل فرآیند در یک پایپ‌لاین خودکار (End-to-End Workflow)

In [8]:
def run_self_correcting_agent(input_text: str):
    print("=" * 60)
    print(" اجرای پایپ‌لاین خودکار بازاندیشی (Reflection Pipeline)")
    print("=" * 60)
    
    # گام ۱: تولید اولیه
    print("\n [گام اول: تولید پیش‌نویس اولیه V1]")
    draft = generate_v1(input_text)
    print(draft)
    
    # گام ۲: اعتبارسنجی در محیط پایتون
    print("\n [گام دوم: اعتبارسنجی با Pydantic]")
    ok, res = validate_json(draft)
    
    if ok:
        print(" خروجی در گام اول معتبر بود.")
        return res
    
    print(" خطای اعتبارسنجی شناسایی شد:\n", res)
    
    # گام ۳: بازاندیشی و اصلاح
    print("\n [گام سوم: بازاندیشی و اصلاح با فیدبک - V2]")
    refined = refine_with_reflection(input_text, draft, res)
    print(refined)
    
    # گام ۴: تایید نهایی
    print("\n [گام چهارم: اعتبارسنجی نهایی V2]")
    ok_v2, final_data = validate_json(refined)
    if ok_v2:
        print(" داده نهایی تایید شد:\n", final_data)
    else:
        print(" نیاز به بررسی بیشتر:", final_data)
    
    print("=" * 60)

# تست پایپ‌لاین کامل
run_self_correcting_agent(sample_text)

🚀 اجرای پایپ‌لاین خودکار بازاندیشی (Reflection Pipeline)

1️⃣ [گام اول: تولید پیش‌نویس اولیه V1]
{
    "customer_name": "سهراب سپهری",
    "email": "sohrab@gmail.com",
    "total_amount": 250.0,
    "items_count": 2
}

2️⃣ [گام دوم: اعتبارسنجی با Pydantic]
✅ خروجی در گام اول معتبر بود.


'{\n  "customer_name": "سهراب سپهری",\n  "email": "sohrab@gmail.com",\n  "total_amount": 250.0,\n  "items_count": 2\n}'

## جمع‌بندی و نتیجه‌گیری
در این پروژه دیدیم که:
1. **پاسخ اولیه (V1)** ممکن است از نظر ظاهری JSON درستی باشد، اما در رعایت قوانین بیزینس و اعتبارسنجی تایپ‌ها شکست بخورد.
2. با ورود **بازخورد خارجی (Pydantic Validation Error)**، مدل توانست دقیقاً متوجه ایراد شود و در **V2** خروجی کاملاً سالم تولید کند.
3. این الگو برای شرکت‌ها و پروژه‌های واقعی جهت **تضمین سلامت داده (Data Reliability)** بسیار حیاتی است.